# Automated LCA Workflow for Exergy Ore Grade Decrease Model

This notebook demonstrates the automated workflow for:
1. Running LCA on electricity production (replacing CSV imports)
2. Automating characterization factor calculations
3. Dynamic method creation with metadata

**Note**: This is a companion to `SurplusEx.ipynb` showing the automated versions of the manual steps.

## Setup
Import libraries and set up Brightway2 environment

In [ ]:
import numpy as np
import pandas as pd
import bw2data as bd
import bw2calc as bc
from mendeleev import element
import sys
sys.path.insert(0, 'src')
from lca_automation import (
    run_lca_electricity,
    get_elementary_flow_contributions,
    calculate_characterization_factors,
    create_method_dynamically,
    run_full_workflow
)

In [ ]:
# Set up Brightway2 project
project_name = 'CERC'
bd.projects.set_current(project_name)

# Verify database is available
print(f'Total elementary flows: {len(ef_df_es)}')
print(f'Current project: {bd.projects.current}')
print(f'Available databases: {list(bd.databases)}')

## Part 1: Run LCA and Get Elementary Flow Contributions

Replace: `IS1_df = pd.read_csv('Results/EsElec/EsS1.csv')`

With: Automated LCA on high-voltage electricity in Spain

In [ ]:
# Run LCA for high-voltage electricity in Spain
lca_es, inventory_es, ef_df_es = run_lca_electricity(country='ES', voltage='high')

# Display elementary flow contributions
print(f'Total elementary flows: {len(ef_df_es)}')
ef_df_es.head()

In [ ]:
# Process elementary flows to match the format used in the original notebook
# Filter for metal-related flows and clean names
metal_keywords = ['aluminium', 'chromium', 'copper', 'iron', 'lead', 'manganese', 
                 'molybdenum', 'nickel', 'zinc', 'uranium', 'phosphorus']

# Filter metal flows
metal_flows_es = ef_df_es[ef_df_es['flow_name'].str.lower().str.contains('|'.join(metal_keywords))]

# Create IS1_df equivalent from LCA results
IS1_df = pd.DataFrame({
    'flow name': metal_flows_es['flow_name'],
    'OG decrease': metal_flows_es['amount']  # Using amount as proxy
)

# Clean flow names (similar to original notebook)
IS1_df_cleaned = IS1_df.copy()
IS1_df_cleaned['flow name'] = IS1_df_cleaned['flow name'].apply(
    lambda x: x.split(',')[0].strip() if isinstance(x, str) else x
)

IS1_df_cleaned.head()

## Part 2: Automated Characterization Factor Calculation

Replace the manual step-by-step calculation with an automated function.

In [ ]:
# Load Ore Grade Decline constants
OGD_df = pd.read_excel("Ore-GradeDeclineConstants.xlsx")
OGD_df.head()

In [ ]:
# Define the functions (same as original)
def OG_ini(alpha, beta, URR, CME):
    xi = np.exp(alpha) * ((URR / CME) - 1) ** beta
    return xi

def Delta_bc(xi, dg):  # dg here is expected to be negative
    R = 8.314
    T = 290.15
    return -R * T * (np.log(xi) + ((1 - xi) / xi) * np.log(1 - xi)) * dg

# Use the automated function
# First, prepare combined_IS_df (this would come from your existing workflow)
# For demonstration, we'll use the cleaned IS1_df
combined_IS_df = IS1_df_cleaned.copy()

# Add OG decrease column if not present
if 'OG decrease' not in combined_IS_df.columns:
    combined_IS_df['OG decrease'] = combined_IS_df['amount'] if 'amount' in combined_IS_df.columns else 0

# Now use the automated calculation
result_df = calculate_characterization_factors(OGD_df, combined_IS_df, option='ERC')

result_df[['flow name', 'Initial Concentration', 'k', 'symbol', 'M', 'CF2_ERC']].head()

## Part 3: Dynamic Method Creation

Replace static method definitions with dynamic ones.

In [ ]:
# Prepare method data from characterization factors
method_data = []
for _, row in result_df.iterrows():
    if pd.notna(row.get('CF2_ERC')):
        # In a real workflow, you'd map flow names to actual flow keys
        # For now, we'll use placeholder keys
        method_data.append((('biosphere', 'placeholder'), row['CF2_ERC']))

# Dynamic method name based on actual data
num_elements = len(result_df[pd.notna(result_df['CF2_ERC'])])
method_name_tuple = (
    "Future Effort method",
    "Spanish Electricity, high voltage",  # Dynamic based on LCA parameters
    f"Surplus Exergy - ERC_dissipative ({num_elements} elements)"  # Dynamic element count
)

# Dynamic metadata
method_metadata = {
    'unit': 'KJ-Eq',
    'description': f'Impact analysis for ore grade decline potential - {num_elements} elements',
    'source': 'Automated workflow based on ReCiPe 2016 constants',
    'version': '1.0',
    'num_cfs': len(method_data),
    'application': 'Input product-system metals characterization'
}

print(f"Method name: {method_name_tuple}")
print(f"Metadata: {method_metadata}")

In [ ]:
# Use the dynamic method creation function
# Note: This will actually create the method in Brightway2
# method_obj = create_method_dynamically(method_name_tuple, method_data, method_metadata)

# For now, just show what would be created
print("\nDynamic method ready to create:")
print(f"  Name: {method_name_tuple}")
print(f"  CFs: {len(method_data)}")
print(f"  Elements: {num_elements}")

## Part 4: Full Automated Workflow

Run everything in one go.

In [ ]:
# Run the full automated workflow
# Note: This requires ecoinvent 3.4 cutoff database to be available

# For Spain, high voltage, ERC option
results_es = run_full_workflow(country='ES', voltage='high', option='ERC')

# For France, high voltage, ERC option
# results_fr = run_full_workflow(country='FR', voltage='high', option='ERC')

# Display results
if results_es:
    print(f"LCA score: {results_es['lca'].score}")
    print(f"Elementary flows: {len(results_es['elementary_flows'])}")
    print(f"Characterization factors calculated: {len(results_es['characterization_factors'])}")
    print(f"Method created: {results_es['method_name']}")

## Summary

### What Changed

1. **CSV Import Replacement**:
   - OLD: `IS1_df = pd.read_csv('Results/EsElec/EsS1.csv')`
   - NEW: `lca, inventory, ef_df = run_lca_electricity(country='ES', voltage='high')`

2. **Calculation Automation**:
   - OLD: Manual step-by-step loops for xi, k, symbols, M, CF2
   - NEW: `result_df = calculate_characterization_factors(OGD_df, combined_IS_df, option='ERC')`

3. **Dynamic Method Creation**:
   - OLD: Hardcoded method names and metadata
   - NEW: Dynamic names based on actual data (`f"Applied to {num_elements} elements"`)

### Benefits

- **Reproducibility**: No more dependency on pre-calculated CSV files
- **Flexibility**: Can run for any country/voltage combination
- **Maintainability**: Cleaner, more modular code
- **Scalability**: Easy to extend to new scenarios

### Next Steps

1. Test the automated workflow with your Brightway2 setup
2. Replace the manual steps in `SurplusEx.ipynb` with calls to these functions
3. Extend the automation to other scenarios (FR, different voltages, etc.)